# How do you score a match between two lists of numbers?


Your music app just played you a song you love, by a band you have never heard of. Somewhere in a data center, two lists of numbers got compared, and the verdict was: same taste.

A song is not one number. It is loud, fast, sad, bright, many qualities at once. Your taste is that same tangle. So before we build anything: how would you boil the match between two many-featured things down to a single number?

We're going to build one answer with our hands, cell by cell.

A taste is an arrow: each entry is one quality. Below, `probe` is you, and each candidate is a song.

Before you run this: just look at the numbers. Which candidate do you guess will end up scoring highest against the probe?

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

# A taste is an arrow: each entry is one quality (say, "loud" and "bright").
probe = np.array([1.0, 0.0])

candidates = {
    "c1": np.array([3.0, 5.0]),
    "c2": np.array([2.0, -1.0]),
    "c3": np.array([0.0, 4.0]),
    "c4": np.array([-2.0, 3.0]),
}

print("probe:", probe)
for name, vec in candidates.items():
    print(name, vec)

Here is the rule for scoring a match: multiply the two arrows entry by entry, then add up the products. One multiply, one add.

Before you run this: does any score come out negative? Say a guess out loud.

In [ ]:
def multiply_and_add(a, b):
    """Multiply entry by entry, then add. That's the whole recipe."""
    return np.sum(a * b)

scores = {name: multiply_and_add(probe, vec) for name, vec in candidates.items()}
print(scores)

for name, vec in candidates.items():
    assert multiply_and_add(probe, vec) == np.dot(probe, vec)

print("c1 has the top score:", scores["c1"] == max(scores.values()))

Now turn the probe. It sweeps through directions while a candidate holds still.

Before you run this: as the probe turns away from a candidate, do you expect its score to fall smoothly, or to jump straight to zero and stay there?

In [ ]:
def rotate(v, angle_rad):
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    rotation = np.array([[c, -s], [s, c]])
    return rotation @ v

target = candidates["c1"]

print(f"{'angle (deg)':>12} | score")
for angle_deg in [0, 30, 60, 90, 120, 150, 180]:
    probe_now = rotate(np.array([1.0, 0.0]), np.deg2rad(angle_deg))
    score = multiply_and_add(probe_now, target)
    print(f"{angle_deg:>12} | {score:6.2f}")

score_at_0 = multiply_and_add(rotate(np.array([1.0, 0.0]), np.deg2rad(0)), target)
score_at_180 = multiply_and_add(rotate(np.array([1.0, 0.0]), np.deg2rad(180)), target)
assert score_at_0 > 0
assert score_at_180 < 0
print("Turn toward it and the score climbs. Turn away and it dies, then goes negative.")

So a big score means two songs are the same, right? Watch one more thing before you decide.

Before you run this: stretch a candidate's length, but keep pointing it the exact same direction. Does its score climb, even though nothing about its direction changed?

In [ ]:
c1 = candidates["c1"]

c1_original_score = multiply_and_add(probe, c1)
c1_stretched = c1 * 2.0
c1_stretched_score = multiply_and_add(probe, c1_stretched)

print("original score:", c1_original_score)
print("stretched score:", c1_stretched_score)

def cosine(a, b):
    return multiply_and_add(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Direction never changed. Cosine similarity (angle-only) proves it.
assert np.isclose(cosine(probe, c1), cosine(probe, c1_stretched))
assert c1_stretched_score > c1_original_score
print("Direction held steady, but the score still climbed. Length cheats the meter.")

Point the probe along a candidate and the sum climbs. Point it across and it dies to zero. Point it against and it goes negative. One multiply and add, and it behaves like a meter with agree, ignore, and oppose on the dial.

**The dot product is a similarity meter.**

Two lists of numbers go in. One honest reading of alignment comes out. We write it `a . b`: multiply pairwise, then add.

Before you run this: guess the sign of each of the three readings below, agree, ignore, oppose, before you see the numbers.

In [ ]:
reference = np.array([3.0, 0.0])
along = np.array([2.0, 0.0])
across = np.array([0.0, 2.0])
against = np.array([-2.0, 0.0])

agree = multiply_and_add(reference, along)
ignore = multiply_and_add(reference, across)
oppose = multiply_and_add(reference, against)

print("agree:", agree, " ignore:", ignore, " oppose:", oppose)

assert agree > 0
assert ignore == 0
assert oppose < 0
print("The dot product is a similarity meter: agree, ignore, oppose.")

Rebuild the meter from memory: two arrows go in, so what two operations turn them into the score? Multiply pairwise, then add. If that came back without a peek, `a . b` is yours now.

While that settles, walk the same meter one aisle over. Swap the songs for a job posting. Score yourself against what the role wants.

Before you run this: do you need to change the function at all to score a job fit instead of a song match?

In [ ]:
# Same meter, different aisle: score yourself against a job posting.
you = np.array([8.0, 5.0, 3.0])   # python hours, sql reps, nights free
role = np.array([6.0, 4.0, 0.0])  # what the role wants

fit_score = multiply_and_add(you, role)
print("fit score:", fit_score)

assert fit_score == np.dot(you, role)
print("Same recipe, new domain. The dot product never asked what the numbers meant.")

Now the honest look. Double `c1`'s length exactly, leave its direction untouched.

Before you run this: does the score double too, or does it just nudge up in some messy way?

In [ ]:
c1_double = c1 * 2.0
score_c1 = multiply_and_add(probe, c1)
score_c1_double = multiply_and_add(probe, c1_double)

print("score:", score_c1, " doubled input score:", score_c1_double)

assert np.isclose(score_c1_double, 2 * score_c1)
print("Double one input, double the output. Size passes straight through the meter.")

## For the walk home

The meter is linear in each arrow you feed it. That clean scaling means a long, loud song can win on sheer size, not on agreement.

What is the cheapest fix that keeps the direction and forgets the size?

Try coding it yourself: divide `probe` and each candidate by their own length (`np.linalg.norm`) before you call `multiply_and_add`. Rerun the stretched `c1` cell with the normalized vectors. Does stretching still change the score?